### This notebook computes a set of indicators from the WaterALLOC database for the 27 basins of IKI Project

**List of Indicators:**
- P13. Water yield: Water production per basin and per square kilometer
- VSS6. Volume of annual water demand for demographic use
- VSS9. Supply reliability: Portion of time during which the water demand is fully met for the agricultural sector
- VSS10. Supply reliability: Portion of time during which the water demand is fully met for the municipal sector
- VSS11. Average deficit, Average magnitude of the supply deficit in the agricultural sector
- VSS12. Average deficit, Average magnitude of the supply deficit in the municipal sector
- VSS13_A, VSS13_P. Index of hydrological stress for the agriculture sector (A) and the municipal sector (P)
- VSB10. Availability of Water by Basin for the Agricultural Sector
- VCA16_A, VCA16_P. Relative frequency of recovery after failure for the agricultural sector (A) and the municiapl sector (P)
- VCA17_A, VCA17_P, VCA17_E. Percentage of net imported water volume relative to the total demand for the agricultural sector (A), the municipal sector (P), and the energy sector (E)
- VCA18. Index of volume of reservoir storage

**Created:** 1/22/2026 by Sophia Bakar (sbakar@rti.org) and Enrique Triana (etriana@rti.org)



In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import sqlite3
import matplotlib.pyplot as plt
import re 
import yaml
from pathlib import Path

#### Seleccion de rutas como funcion del usuario

In [ ]:
# set master path for input data from config file

config_path = Path("../../config.yaml")

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

master_path = Path(config["master_path"])

In [ ]:
subbasins_shapefile = master_path / "Modelacion" / "Grupos_Modelacion" / "GIS_WaterALLOC_General" / "Peru_AHD_with_districts.shp"
db_path = master_path / "Indicadores" / "BD_RiesgoClimatico_IKI.db"
wateralloc_db = master_path / "Modelacion" / "Grupos_Modelacion" / "Resultados" / "BalanceHidrico.sqlite"

In [ ]:
subbasins_gdf = gpd.read_file(subbasins_shapefile).set_index('COMID').to_crs('WGS84')

In [4]:
conn = sqlite3.connect(db_path)

## NEED A NEW APPROACH/WAY TO FLAG FOR THIS

indicators_df = pd.read_sql_query(
    """
    SELECT IndID, TextID
    FROM Indicators
    WHERE SIG_Type = 'Wateralloc DB'
    """,
    conn
)

conn.close()

In [5]:
# Connect to Indicators DB and get available scenarios
conn = sqlite3.connect(db_path)

wa_scenarios_df = pd.read_sql_query(
    """
    SELECT WaScnID, WaScnName
    FROM WaScenarios
    ORDER BY WaScnID
    """,
    conn
)

conn.close()

# For now: only baseline and first future
scenario_ids = wa_scenarios_df.loc[
    wa_scenarios_df['WaScnID'].isin([1, 3]), 'WaScnID'
].tolist()

# for all scenarios:
#scenario_ids = wa_scenarios_df['WaScnID'].tolist()

wa_scenarios_df

,WaScnID,WaScnName
0,1,CC_CMIP6_85_2050
1,3,Linea_Base_2020
2,4,Linea_Base_2020_Embalses


### Process all the WaterALLOC Scenarios
Los escenarios en WaterALLOC deben corresponder a un escenario en la base de datos de indicadores. Es esta seccion procesamos todos los escenarios en la base de datos vinculando con el escenario correspondiente en la base de datos de indicadores.

Este calculo necesita un indicador unico en la base de datos de indicadores para las combinaciones de  escenarios de indicadores y WaterALLOC.  
#### Method
Using SQL we attach the indicators database and execute an insert query in the indicators database using the processed data from the WaterALLOC scenarios database.  

Only the COMIDs processed to the WaterALLOC database are available.  The risk calculation should handle the missing COMID values.

#### Calculation Notes
The indicator for each COMID is queried as the .  

In [6]:
# Connect to WaterALLOC database
conn_wa = sqlite3.connect(wateralloc_db)
cursor = conn_wa.cursor()
# Attach RiesgoDB database
attach_query = fr"ATTACH DATABASE '{db_path}' AS RiesgoDB;"
cursor.execute(attach_query)

In [8]:
# Create temp table with ALL COMIDs from shapefile
subbasins_gdf.reset_index()[['COMID']].astype(int).to_sql(
    "AllSubbasinCOMIDs",
    conn_wa,
    if_exists="replace",
    index=False
)

pd.read_sql("SELECT * FROM AllSubbasinCOMIDs LIMIT 10", conn_wa)

,COMID
0,311153600
1,310832400
2,310825700
3,310282400
4,308754600
5,307736500
6,307078200
7,307183500
8,306538400
9,306599700


In [ ]:
subbasins_area_df = (
    subbasins_gdf.reset_index()[['COMID', 'AREASQKM']]
    .astype({'COMID': int})
)

subbasins_area_df.to_sql(
    "SubbasinArea",
    conn_wa,
    if_exists="replace",
    index=False
)

In [ ]:
## Create/update views that use existing tables to simplify indicator calculations
# View for Import/Export calculation
create_view_query = """
DROP VIEW IF EXISTS "main"."ImportExport fraction por COMID";
"""
cursor.execute(create_view_query)

create_view_query = """
CREATE VIEW "ImportExport fraction por COMID" AS SELECT a.RunID,a.Cuenca, a.COMID, 
	sum(importExport) AS NetImport,sum(importExport)/avg(TotAfluencia) AS ImportFract 
FROM (
	SELECT RunID,Cuenca,COMID, avg(Volumen) AS ImportExport
	FROM "WAMSS_Importacion y Exportacion anual por COMID"
	WHERE [Tipo] = 'Importacion'
	GROUP BY RunID,Cuenca, COMID
	UNION 
	SELECT RunID,Cuenca,COMID, avg(-Volumen) As ImportExport
	FROM "WAMSS_Importacion y Exportacion anual por COMID"
	WHERE [Tipo] = 'Exportacion'
	GROUP BY RunID,Cuenca, COMID
) as a
LEFT JOIN (
	SELECT RunID,Cuenca,avg(Afluencia) AS [TotAfluencia]
	FROM "WAMSS_Oferta anual por tipo por cuenca"
	GROUP BY RunID,Cuenca
) AS b ON b.RunID = a.RunID AND b.Cuenca = a.Cuenca 
GROUP BY a.RunID,a.Cuenca, a.COMID
"""
cursor.execute(create_view_query)

# view for P13 (water production per COMID normalized by area)
cursor.execute("""DROP VIEW IF EXISTS "OfertaNorm por COMID";""")

cursor.execute("""
CREATE VIEW "OfertaNorm por COMID" AS
SELECT 
    annual.RunID,
    annual.COMID,
    AVG(annual.OfertaTot_anual) / area.AREASQKM AS OfertaNorm
FROM (
    SELECT 
        RunID,
        COMID,
        Año,
        SUM(Afluencia) AS OfertaTot_anual
    FROM "WAMSS_Oferta anual por tipo por COMID"
    WHERE TipoAfluencia <> 'Recarga'
    GROUP BY RunID, COMID, Año
) AS annual
JOIN SubbasinArea area
    ON area.COMID = annual.COMID
GROUP BY annual.RunID, annual.COMID
""")

# view for VCA18
cursor.execute("""DROP VIEW IF EXISTS "ReservoirStorage por COMID";""")

cursor.execute("""
CREATE VIEW "ReservoirStorage por COMID" AS
SELECT 
    b.WaScnID,
    a.RunID,
    a.COMID,
    AVG(a.Almacenamiento) AS Storage_Medio
FROM "WAMSS_Volumen promedio en embalses por COMID" a
JOIN WAMMS_RunsInfo b 
    ON a.RunID = b.RunID
GROUP BY b.WaScnID, a.RunID, a.COMID
""")


In [ ]:
for waScn_ID in scenario_ids:

    print(f"\nProcessing WA scenario WaScnID={waScn_ID}")

    # Get RunIDs for this scenario
    run_ids_list = [row[0] for row in cursor.execute(
        f"SELECT RunID FROM WAMMS_RunsInfo WHERE WaScnID = {waScn_ID}"
    ).fetchall()]
    print(f"\tRunIDs for this scenario: {run_ids_list}")

    for ind_row in indicators_df.itertuples(index=False):
        indID = ind_row.IndID
        print(f"\tProcessing indicator {ind_row.TextID} (IndID={indID})")

        default_fill = 0  # default

        if ind_row.TextID in ["VSS9", "VSS10"]:
            default_fill = 1

        if ind_row.TextID == "VSS9":
            source_table = "[WAMSS_Confiabilidad y Deficit por COMID]"
            value_calc = "b.[Confiabilidad]"
            and_where = "AND b.Sector = 'Agrario'"

        elif ind_row.TextID == "VSB10":
            source_table = "[WAMSS_Balance por COMID (+Indice de estres)]"
            value_calc = "(b.[Oferta Local Sup] + b.[Oferta Entrada] - b.[Dem Local Sup])"
            and_where = ""

        elif ind_row.TextID == "VSS10":
            source_table = "[WAMSS_Confiabilidad y Deficit por COMID]"
            value_calc = "b.[Confiabilidad]"
            and_where = "AND b.Sector = 'Poblacional'"

        elif ind_row.TextID == "VSS11":
            source_table = "[WAMSS_Confiabilidad y Deficit por COMID]"
            value_calc = "b.[DeficitProm]"
            and_where = "AND b.Sector = 'Agrario'"

        elif ind_row.TextID == "VSS12":
            source_table = "[WAMSS_Confiabilidad y Deficit por COMID]"
            value_calc = "b.[DeficitProm]"
            and_where = "AND b.Sector = 'Poblacional'"

        elif ind_row.TextID in ["VSS13_A", "VSS13_P"]:
            source_table = "[WAMSS_Balance por COMID (+Indice de estres)]"
            value_calc = "b.[IndiceEstres_Sup]"
            and_where = ""

        elif ind_row.TextID in ["VCA16_A", "VCA16_P"]:
            source_table = "[WAMSS_Resiliencia por COMID y Sector]"
            value_calc = "b.[frequency_1_to_0]"
            and_where = (
                "AND b.Sector = 'Agrario'" if ind_row.TextID.endswith("_A") else "AND b.Sector = 'Poblacional'"
            )

        elif ind_row.TextID in ["VCA17_A", "VCA17_P", "VCA17_E"]:
            source_table = "[ImportExport fraction por COMID]"
            value_calc = "b.[ImportFract]"
            and_where = ""

        elif ind_row.TextID == "VSS6":
            source_table = "[WAMSS_Demanda anual promedio por tipo de demanda por COMID]"
            value_calc = "CASE WHEN b.Demanda = 0 THEN 0 ELSE b.Suministro / b.Demanda END"
            and_where = "AND b.Sector = 'Poblacional'"

        elif ind_row.TextID == "P13":
            source_table = "[OfertaNorm por COMID]"
            value_calc = "b.[OfertaNorm]"
            and_where = ""
    
        elif ind_row.TextID == "VCA18":
            source_table = "[ReservoirStorage por COMID]"
            value_calc = "b.[Storage_Medio]"
            and_where = ""

        else:
            print(f"\t\tSkipping undefined indicator {ind_row.TextID}")
            continue

        # delete previous values for this scenario and indicator
        cursor.execute(f"""
            DELETE FROM [RiesgoDB].IndValues_WaALLOC
            WHERE WaScnID = {waScn_ID}
              AND IndID  = {indID};
        """)

        # check how many rows we have in the source table for this scenario and indicator
        count_source_rows = cursor.execute(f"""
            SELECT COUNT(*)
            FROM {source_table} AS b
            WHERE b.RunID IN ({','.join(map(str, run_ids_list))})
            {and_where};
        """).fetchone()[0]
        print(f"\t\tRows in source table for this indicator & scenario: {count_source_rows}")

        default_fill = 1 if ind_row.TextID in ["VSS9", "VSS10"] else 0

        # insert new calculated values
        insert_query = f"""
            INSERT INTO [RiesgoDB].IndValues_WaALLOC (WaScnID, IndID, COMID, Value)
            SELECT
                {waScn_ID} AS WaScnID,
                {indID} AS IndID,
                c.COMID,
                COALESCE(v.Value, {default_fill}) AS Value
            FROM AllSubbasinCOMIDs AS c
            LEFT JOIN (
                SELECT
                    b.COMID,
                    AVG({value_calc}) AS Value
                FROM {source_table} AS b
                WHERE b.RunID IN ({','.join(map(str, run_ids_list))})
                {and_where}
                GROUP BY b.COMID
            ) AS v
            ON v.COMID = c.COMID;
        """
        cursor.execute(insert_query)

        n_rows = cursor.execute(
            f"""
            SELECT COUNT(*)
            FROM [RiesgoDB].IndValues_WaALLOC
            WHERE WaScnID = {waScn_ID}
              AND IndID  = {indID};
            """
        ).fetchone()[0]

        print(f"\t\tInserted {n_rows} rows")

conn_wa.commit()


Processing WA scenario WaScnID=1
	RunIDs for this scenario: [1, 4, 6]
	Processing indicator VSB10 (IndID=310)
		Rows in source table for this indicator & scenario: 250236
		Inserted 3655 rows
	Processing indicator VCA16_A (IndID=5161)
		Rows in source table for this indicator & scenario: 186
		Inserted 3655 rows
	Processing indicator VCA16_P (IndID=5162)
		Rows in source table for this indicator & scenario: 59
		Inserted 3655 rows
	Processing indicator VCA17_A (IndID=5171)
		Rows in source table for this indicator & scenario: 2
		Inserted 3655 rows
	Processing indicator VCA17_P (IndID=5172)
		Rows in source table for this indicator & scenario: 2
		Inserted 3655 rows
	Processing indicator VCA17_E (IndID=5173)
		Rows in source table for this indicator & scenario: 2
		Inserted 3655 rows
	Processing indicator VSS9 (IndID=409)
		Rows in source table for this indicator & scenario: 186
		Inserted 3655 rows
	Processing indicator VSS10 (IndID=410)
		Rows in source table for this indicator & sc